<a href="https://colab.research.google.com/github/nmanne-bit/nlp-ml-project/blob/main/Copy_of_ExplainableAIC_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

### Install the dependencies

!pip install langchain langchain-community faiss-cpu sentence-transformers transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 865.3 kB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
#### Creat Knowledge Base
from langchain_core.documents import Document

docs = [
    Document(page_content="AI is transforming healthcare using predictive analytics."),
    Document(page_content="Machine learning helps farmers predict crop yield."),
    Document(page_content="NLP enables chatbots to understand human language."),
    Document(page_content="BERT is a powerful transformer model for text understanding."),
]

In [ ]:
### ### Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_13211/3342665012.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#### Vector Store ( FAISS )
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
from transformers import pipeline

### Local LLM
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=100   # 🔥 FIX
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
#### Custom RAG function
def rag_chat(query):
    # Step 1: Retrieve docs
    results = vectorstore.similarity_search_with_score(query, k=3)

    context = " ".join([doc.page_content for doc, _ in results])

    # Step 2: Generate answer
    prompt = f"""
    Answer the question using the context below.
    Context: {context}
    Question: {query}
    Answer:
    """

    output = generator(prompt)[0]["generated_text"]

    return output, results

In [ ]:
### chat Example

query = "How is AI used in healthcare?"

answer, docs_used = rag_chat(query)

print("Answer:", answer)

print("\n--- Explainability ---")
for doc, score in docs_used:
    print("Doc:", doc.page_content)
    print("Score:", score)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: 
    Answer the question using the context below.
    Context: AI is transforming healthcare using predictive analytics. NLP enables chatbots to understand human language. BERT is a powerful transformer model for text understanding.
    Question: How is AI used in healthcare?
    Answer:
     Answer the question using the context below.
      Answer the question using the context below.
      Answer the question using the context below.
       Answer the question using the context below.
       Answer the question using the context below.
        Answer the question using the context below.
                 

--- Explainability ---
Doc: AI is transforming healthcare using predictive analytics.
Score: 0.7026427
Doc: NLP enables chatbots to understand human language.
Score: 1.3174653
Doc: BERT is a powerful transformer model for text understanding.
Score: 1.5148888


In [ ]:
#### Evaluation metrics
### Retrieval Accuracy (Hit Rate)
def hit_rate(query, expected_keyword):
    results = vectorstore.similarity_search(query, k=3)

    for doc in results:
        if expected_keyword.lower() in doc.page_content.lower():
            return 1
    return 0

# Example
print("Hit Rate:", hit_rate("AI in healthcare", "healthcare"))

Hit Rate: 1


In [ ]:
#### Precision@k

def precision_at_k(query, keyword, k=3):
    results = vectorstore.similarity_search(query, k=k)

    relevant = 0
    for doc in results:
        if keyword.lower() in doc.page_content.lower():
            relevant += 1

    return relevant / k

print("Precision@3:", precision_at_k("crop prediction", "crop"))

Precision@3: 0.3333333333333333


In [ ]:
### Cosine Similarity score

from sklearn.metrics.pairwise import cosine_similarity

def similarity_score(query):
    query_vec = embeddings.embed_query(query)
    doc_vecs = embeddings.embed_documents([d.page_content for d in docs])

    sims = cosine_similarity([query_vec], doc_vecs)
    return sims

print("Similarity:", similarity_score("AI healthcare"))

Similarity: [[0.68826666 0.20290169 0.28057889 0.21219545]]


In [ ]:
#### Response Length/ Coherence Proxy
def response_length(answer):
    return len(answer.split())

print("Response Length:", response_length(answer))

Response Length: 81
